In [ ]:
import oncophylo as op
import numpy as np
import pandas as pd
import anndata as ad
import os, sys
import matplotlib.pyplot as plt
from utils import add_population_nodes

In [ ]:
path = os.path.join(os.getcwd(), "data", "AML")

In [ ]:
sample1 = "AML-21-001"
sample2 = "AML-21-002"

In [ ]:
sample1_variants_df = pd.read_csv(os.path.join(path, f"{sample1}_variants.csv"), header=0, index_col=0)
sample1_regions_df = pd.read_csv(os.path.join(path, f"{sample1}_regions.csv"), header=None, index_col=0)

sample2_variants_df = pd.read_csv(os.path.join(path, f"{sample2}_variants.csv"), header=0, index_col=0)
sample2_regions_df = pd.read_csv(os.path.join(path, f"{sample2}_regions.csv"), header=None, index_col=0)

In [ ]:
# rename FLT3 intronic
rename = sample2_variants_df.loc[:,"NAME"].values
rename[2:4] = ["FLT3-ITD_1", "FLT3-ITD_2"]
sample2_variants_df.loc[:,"NAME"] = rename

In [ ]:
data = [(sample1_variants_df, sample1_regions_df), (sample2_variants_df, sample2_regions_df)]

In [ ]:
character_matrix, variant_reads_df, total_reads_df, meta_df, regions_df, cell_samples = op.ul.preprocess_longitudinal(data)

In [ ]:
sol_COMPASS = op.tl.solver.COMPASS(character_matrix, variant_reads_df, total_reads_df, regions_df, meta_df, remove_temp_dir=False)

In [ ]:
op.pl.show_tree(sol_COMPASS[op.ul.DATA.MUTATION_TREE])

In [ ]:
sol_LoPhy = op.tl.solver.LoPhy(character_matrix.replace(2,1).replace(3,-1), 
                                         variant_reads_df, 
                                         total_reads_df,                        
                                         regions_df, 
                                         meta_df,
                                         cell_samples=cell_samples,
                                         remove_temp_dir=False,
                                         seed=0)

In [ ]:
T = sol_LoPhy[op.ul.DATA.MUTATION_TREE].copy()

In [ ]:
_, T = op.io.load_dot(os.path.join(os.getcwd(), "LoPhy_output", "out_ml0.gv"), 
                           _type="cell_tree")

In [ ]:
add_population_nodes(T, regions_df, meta_df, cell_samples)

In [ ]:
T.nodes["Clone_0"]["label"] = "root"

# update node label format
for n in T.nodes:
    print(n)
    T.nodes[n]["label"] = T.nodes[n]["label"].replace("_p", " p.")
    T.nodes[n]["label"] = T.nodes[n]["label"].replace("_", " ")

In [ ]:
op.pl.show_tree(T, save_path=os.path.join(os.getcwd(), "..", "paper", "real_results", "AML21", "LoPhy_AML21_tree.svg"))